# Data Quality Checks

This notebook checks the initial dataset preview for missing review text, empty reviews and duplicate records.

In [3]:
from pathlib import Path
import gzip
import json

import pandas as pd

RAW_DATA_PATH = Path("../data/raw/Video_Games_5.json.gz")
PREVIEW_ROWS = 10_000

records = []

with gzip.open(RAW_DATA_PATH, "rt", encoding="utf-8") as file:
    for index, line in enumerate(file):
        if index >= PREVIEW_ROWS:
            break
        records.append(json.loads(line))

reviews_preview = pd.DataFrame(records)

In [5]:
missing_review_text = reviews_preview["reviewText"].isna().sum()
empty_review_text = reviews_preview["reviewText"].fillna("").str.strip().eq("").sum()

duplicate_columns = [
    "reviewerID",
    "asin",
    "reviewText",
    "overall",
    "reviewTime"
]

duplicate_columns = [
    column for column in duplicate_columns
    if column in reviews_preview.columns
]

duplicate_records = reviews_preview.duplicated(
    subset=duplicate_columns
).sum()

duplicate_review_text = reviews_preview.duplicated(
    subset=["reviewText"]
).sum()

print(f"Rows in preview: {len(reviews_preview):,}")
print(f"Missing review text: {missing_review_text:,}")
print(f"Empty review text: {empty_review_text:,}")
print(f"Duplicate records: {duplicate_records:,}")
print(f"Duplicate review text: {duplicate_review_text:,}")

Rows in preview: 10,000
Missing review text: 0
Empty review text: 0
Duplicate records: 2
Duplicate review text: 510


## Initial observations

The 10,000-row preview contained no missing or empty review text. Only two records were duplicated when using reviewer, product, review text, rating and date fields together.

There were 510 repeated review-text values. Repeated text alone will not automatically be removed because short or generic reviews may legitimately be written by different users for different products. The cleaning stage will remove only duplicate records based on the selected record fields.